In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gym
import numpy as np
import random

In [ ]:
class DQNAgent(nn.Module):
    def __init__(self, state_size, action_size):
      super(DQNAgent, self).__init__()
      self.fc1 = nn.Linear(state_size, 120)
      self.relu = nn.ReLU()
      self.fc2 = nn.Linear(120, 84)
      self.fc3 = nn.Linear(84, action_size)
    def forward(self, x):
      x = self.relu(self.fc1(x))
      x = self.relu(self.fc2(x))
      x = self.fc3(x)
      return x

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
class ReplayMemory:
    def __init__(self, capacity, state_size):
        self.capacity = capacity
        self.states = np.zeros((capacity, state_size), dtype=np.float32)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.next_states = np.zeros((capacity, state_size), dtype=np.float32)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.done = np.zeros(capacity, dtype=np.int64)
        self.index = 0
        self.size = 0

    def push(self, state, action, next_state, reward, done):
        self.states[self.index] = state
        self.actions[self.index] = action
        self.next_states[self.index] = next_state
        self.rewards[self.index] = reward
        self.done[self.index] = done
        self.index = (self.index + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        indices = np.random.choice(self.size, batch_size, replace=False)
        return (
            torch.tensor(self.states[indices], dtype=torch.float32),
            torch.tensor(self.actions[indices], dtype=torch.long),
            torch.tensor(self.next_states[indices], dtype=torch.float32),
            torch.tensor(self.rewards[indices], dtype=torch.float32),
            torch.tensor(self.done[indices], dtype=torch.int64),
        )

    def __len__(self):
        return self.size

def linear_schedule(start_e: float, end_e: float, duration: int, t: int):
    slope = (end_e - start_e) / duration
    return max(slope * t + start_e, end_e)

In [ ]:
class DQNTrainer:
    def __init__(self, state_size, action_size):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.state_size = state_size
        self.action_size = action_size
        self.memory = ReplayMemory(10000, state_size)
        self.total_timesteps = 500000
        self.gamma = 0.99    # discount rate
        self.start_epsilon = 1.0  # exploration rate
        self.epsilon_min = 0.01
        self.epsilon = self.start_epsilon
        self.exp_fraction = 0.5
        self.learning_starts = 10000
        self.train_freq = 10
        self.batch_size = 128
        self.target_update = 500
        self.Q_net = DQNAgent(state_size, action_size).to(self.device)
        self.target_net = DQNAgent(state_size, action_size).to(self.device)
        self.target_net.load_state_dict(self.Q_net.state_dict())
        self.target_net.eval()
        self.optimizer = optim.Adam(self.Q_net.parameters(), lr=1e-4)

    def select_action(self, state, eps):
        if np.random.rand() <= eps:
          return torch.tensor([[random.randrange(self.action_size)]], device=self.device, dtype=torch.long)
        else:
          return torch.argmax(self.Q_net(state), axis=1)

    def optimize_model(self):
        if len(self.memory) < self.batch_size:
            return
        states, actions, next_states, rewards, done = self.memory.sample(self.batch_size)
        with torch.no_grad():
          next_state_values, _ = self.target_net(next_states).max(dim=1)
          expected_next_state_values = rewards + (1-done)*self.gamma * next_state_values
        state_action_values = self.Q_net(states).gather(1, actions.unsqueeze(1)).squeeze()
        loss = F.mse_loss(state_action_values, expected_next_state_values)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()

    def train(self, env):
        state = env.reset()
        state = torch.tensor([state], device=self.device, dtype=torch.float32)
        for step in range(self.total_timesteps+1):
            action = self.select_action(state, self.epsilon)
            next_state,  reward, done, _ = env.step(action.item())
            next_state = torch.tensor([next_state], device=self.device, dtype=torch.float32)
            self.memory.push(state, action, next_state, reward, done)
            state = next_state

            if step >= self.learning_starts and step % self.train_freq == 0:
              self.optimize_model()

            if step % self.target_update == 0:
              self.target_net.load_state_dict(self.Q_net.state_dict())

            if done:
              state = env.reset()
              state = torch.tensor([state], device=self.device, dtype=torch.float32)

            if step % 10000 == 0:
                eval_reward = self.evaluate()
                print("Step: {}, Evaluation Reward: {}".format(step, eval_reward))
            self.epsilon = linear_schedule(self.start_epsilon, self.epsilon_min, self.exp_fraction * self.total_timesteps, step)

    def evaluate(self, render_mode=None, episodes=10):
        env = gym.make('CartPole-v1', render_mode=render_mode)
        total_reward = 0
        for _ in range(episodes):
            state = env.reset()
            state = torch.tensor([state], device=self.device, dtype=torch.float32)
            episode_reward = 0
            done = False
            while not done:
                action = self.select_action(state, 0.05)
                next_state, reward, done, _ = env.step(action.item())
                if render_mode != None:
                  env.render()
                episode_reward += reward
                state = torch.tensor([next_state], device=self.device, dtype=torch.float32) if not done else None
            total_reward += episode_reward
        return total_reward / episodes

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [ ]:
if __name__ == "__main__":
    env = gym.make('CartPole-v1')
    state_size = env.observation_space.shape[0]
    action_size = env.action_space.n
    agent = DQNTrainer(state_size, action_size)
    agent.train(env)


/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/utils/passive_env_checker.py:241: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Step: 0, Evaluation Reward: 37.7
Step: 10000, Evaluation Reward: 49.0
Step: 20000, Evaluation Reward: 10.0
Step: 30000, Evaluation Reward: 68.8
Step: 40000, Evaluation Reward: 148.9
Step: 50000, Evaluation Reward: 220.7
Step: 60000, Evaluation Reward: 382.3
Step: 70000, Evaluation Reward: 242.4
Step: 80000, Evaluation Reward: 323.1
Step: 90000, Evaluation Reward: 342.9
Step: 100000, Evaluation Reward: 321.8
Step: 110000, Evaluation Reward: 302.6
Step: 120000, Evaluation Reward: 339.9
Step: 130000, Evaluation Reward: 379.5
Step: 140000, Evaluation Reward: 343.1
Step: 150000, Evaluation Reward: 405.9
Step: 160000, Evaluation Reward: 351.9
Step: 170000, Evaluation Reward: 411.1
Step: 180000, Evaluation Reward: 439.5
Step: 190000, Evaluation Reward: 351.4
Step: 200000, Evaluation Reward: 319.7
Step: 210000, Evaluation Reward: 325.4
Step: 220000, Evaluation Reward: 371.5
Step: 230000, Evaluation Reward: 432.0
Step: 240000, Evaluation Reward: 473.0
Step: 250000, Evaluation Reward: 464.9
Step